# Get to Know Dataset: Infant 3T/7T Precision Imaging

This notebook serves as a basic introduction to the [Infant 3T/7T Precision Imaging](https://registry.opendata.aws/infant-3t-7t-precision-mri) dataset. This accompanies the paper [Precision functional imaging in infants using multi-echo fMRI at 7T](https://www.biorxiv.org/content/10.1101/2025.11.09.687453v1). Accompanying code and more information can be found at the [infant_3T_7T_precision_imaging](https://github.com/DCAN-Labs/infant_3T_7T_precision_imaging) GitHub repository. The dataset is hosted on the [Registry of Open Data on AWS](https://registry.opendata.aws/).

Data in this repository are organized following [BIDS standard](https://bids.neuroimaging.io/).

For each of the six "Precision Baby" (PB) subjects included in this manuscript, input data as well as data derivatives are available. Shared data for each subject is organized in the following way:


```bash
PB0XX
├── BIDSinput
│   ├── dataset_description.json
│   └── sub-PB0XX
│       └── ses-MENORDIC
│           ├── anat
│           ├── fmap
│           └── func
├── NiBabiesDerivatives
│   ├── dataset_description.json
│   ├── sub-PB0XX_ses-MENORDIC.html
│   ├── logs
│   └── sub-PB0XX
│       ├── figures
│       └── ses-MENORDIC
│           ├── anat
│           ├── fmap
│           └── func
├── XCPDderivatives
    ├── dataset_description.json
    ├── sub-PB00XX.html
    ├── sub-PB0XX_ses-MENORDIC_executive_summary.html
    ├── atlases
    ├── logs
    └── sub-PB0XX
        ├── figures
        ├── log
        └── ses-MENORDIC
            ├── anat
            ├── fmap
            └── func
```



First we will import the Python libraries required throughout this notebook.

In [ ]:
# This notebook requires the following library:
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# boto3 >= 1.38.23

# Import the libraries required for this notebook
# Built-ins
import json
from pprint import pprint
# Installed libraries
import boto3
from botocore import UNSIGNED
from botocore.config import Config

Next, we will define the location of our dataset, create our boto3 S3 client, and list the S3 bucket objects. Here we see that the data are provided as a single zip data file for each study participant:

In [ ]:
# Location of the S3 bucket for this dataset
bucket = "infant-3t-7t-precision-mri"

# List the top level of the bucket using boto3. Because this is a public bucket, we don't need to sign requests.
# Here we set the signature version to unsigned, which is required for public buckets.
bucket = "infant-3t-7t-precision-mri"

# Connect to the public S3 bucket without credentials
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))

# List bucket contents
response = s3.list_objects_v2(Bucket=bucket)

for item in response.get("Contents", []):
    print(item["Key"])

To get a deeper sense of the deeper structure of the dataset, we will next need to download and unpack one of the zip files. 

<!-- CHECK IF OU CAN LIST S3 zip files contents -->

In [ ]:
import boto3
import io
import zipfile
from botocore import UNSIGNED
from botocore.client import Config

bucket = "infant-3t-7t-precision-mri"
zip_key = "dataPB022.zip"

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))

# Download ZIP into memory
response = s3.get_object(Bucket=bucket, Key=zip_key)
zip_bytes = response["Body"].read()

# List contents
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    for name in z.namelist():
        print(name)


The data directory of every study participants contains BIDS input data, minimally preprocessed derivatives and derivatives from post-processing.

BISD input data has already undergone thermal noise removal with NORDIC (see for example [Vizioli et al. 2021](https://www.nature.com/articles/s41467-021-25431-8)). 

Preprocessed derivatives have been optained with NiBabies version 25.0.1 ([Goncalves & Moser et al. 2025, bioRxiv](https://www.biorxiv.org/content/10.1101/2025.05.14.654069v1)). 

Postprocessed derivatives with XCP-D version 0.10.5 ([Mehta et al., 2024](https://direct.mit.edu/imag/article/doi/10.1162/imag_a_00257/123715/XCP-D-A-robust-pipeline-for-the-post-processing-of)). 

All data (3T and 7T) are combined withing the respective 'func' directories. They are differentiated using the BIDS format `acq` label (`acq-3T2mm`; `acq-7T16mm` (for 1.6mm resolution) and `acq-7T125mm` (for 1.25mm resolution)).

Please refer to the manuscript for a more detailed description of the input data and preprocessing steps. 



Data can be visualized using tools such as FSLeyes or Conectome Workbench. 
Below is an example visualization of raw images from the four-echo 3T sequence and the two different three-echo 7T sequences within PB0020 (*Suppl. Figure 1 in the publication*).

![Raw ME 7T & 3T snapshots](rawME-images.png)

When unpacking the data, Supplementary Table 1 from the manuscript (below) additionally help orient regarding what's available per participant. 

![](table1.png)

Using this dataset, we can answer questions around the impact of data quality (in terms of spatial precision and reliability) on our ability to precisely measure functional brain organization in infants. Our initial findings are summarized in the manuscript [Precision functional imaging in infants using multi-echo fMRI at 7T](https://www.biorxiv.org/content/10.1101/2025.11.09). 


The initial analyses have not exhausted what can possibly be discovered with multi-echo ultra-high field infant fMRI data. Further oportunities encompass for example echo specific analyses or analyses of multi-echo derived T2* and R2* in relation with iron accumulation in the developing brain. 